# DiffDock vs SigmaDock — Calibration Comparison (PoseBusters)

Compares **pose distribution calibration** of DiffDock (40 samples, confidence-ranked)
against SigmaDock under four conditions (ODE-25, SDE-10, SDE-25, SDE-50; 40 seeds each,
heuristic-ranked) using:

- **TARP** (symRMSD, K=10): ECP curves; diagonal = perfect calibration
- **MIRA** (symRMSD, num_runs=100): scalar per-complex; `mira_null(40) ≈ 0.683` = perfect
- **Group TARP / MIRA** (K=100 / num_runs=100): broken down by Translation / Rotation / Torsion DOF
- **Top-1 RMSD accuracy**: fraction of complexes where rank-1 pose has RMSD < 2 Å

Cells degrade gracefully if a file is not yet available.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

from molcalib.tarp import ecp_from_fractions, bootstrap_ecp, plot_ecp
from molcalib.mira import mira_null

S         = 40
MIRA_NULL = mira_null(S)
N_BOOT    = 200
GROUPS    = ['translation', 'rotation', 'torsion']
GROUP_LABELS = {
    'translation': 'Translation (R³)',
    'rotation':    'Rotation (SO(3))',
    'torsion':     'Torsion (T^k)',
}

print(f'MIRA null (S={S}): {MIRA_NULL:.4f}')

## 1. Paths & conditions

In [ ]:
SD_BASE    = '/home/qf226/rds/hpc-work/results/SigmaDock/sigmadock_pb_sde/results/posebusters'
DD_METRICS = '/home/qf226/rds/hpc-work/results/DiffDock/pb_evaluate_v2_merged/metrics'

CONDITIONS = {
    'ode_25':  {'label': 'SigmaDock ODE-25', 'color': '#4CAF50', 'ls': '-'},
    'steps10': {'label': 'SigmaDock SDE-10', 'color': '#2196F3', 'ls': '-'},
    'steps25': {'label': 'SigmaDock SDE-25', 'color': '#9C27B0', 'ls': '-'},
    'steps50': {'label': 'SigmaDock SDE-50', 'color': '#FF9800', 'ls': '-'},
}
DD_LABEL = 'DiffDock'
DD_COLOR = '#F44336'
DD_LS    = '--'

## 2. Load data

In [ ]:
def _npy(path):
    """Load .npy file; return None if missing."""
    return np.load(path, allow_pickle=True) if os.path.exists(path) else None

def _npz_key(path, key):
    """Load one key from .npz; return None if missing."""
    if not os.path.exists(path):
        return None
    d = np.load(path, allow_pickle=True)
    return d[key] if key in d else None

def _json(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

# ── SigmaDock ─────────────────────────────────────────────────────────────────
sd = {}
for cond in CONDITIONS:
    m   = f'{SD_BASE}/{cond}/metrics'
    ge  = f'{m}/group_eval'
    npz = f'{m}/mira_tarp.npz'
    d = {
        'mira_scores': _npz_key(npz, 'mira_scores'),
        'tarp_f':      _npz_key(npz, 'tarp_f_matrix'),
        'top1_rmsd':   _npy(f'{m}/top1_rmsd.npy'),
        'pb_json':     _json(f'{m}/posebusters_results_pb.json'),
    }
    for g in GROUPS:
        d[f'tarp_{g}'] = _npy(f'{ge}/tarp_fractions_{g}.npy')
        d[f'mira_{g}'] = _npy(f'{ge}/mira_scores_{g}.npy')
    sd[cond] = d

# ── DiffDock ──────────────────────────────────────────────────────────────────
dd = {
    'mira_scores': _npy(f'{DD_METRICS}/mira_scores_symrmsd.npy'),
    'tarp_f':      _npy(f'{DD_METRICS}/tarp_fractions_symrmsd_K10.npy'),   # pending
    'tarp_K1':     _npy(f'{DD_METRICS}/tarp_fractions_symrmsd_K1.npy'),    # fallback
    'top1_rmsd':   _npy(f'{DD_METRICS}/top1_rmsd.npy'),
    'pb_json':     _json(f'{DD_METRICS}/posebusters_results_pb.json'),
}
for g in GROUPS:
    dd[f'tarp_{g}'] = _npy(f'{DD_METRICS}/group_eval/tarp_fractions_{g}.npy')
    dd[f'mira_{g}'] = _npy(f'{DD_METRICS}/group_eval/mira_scores_{g}.npy')

In [ ]:
# ── Readiness table ───────────────────────────────────────────────────────────
def _r(x): return '✓' if x is not None else '–'

keys = [
    ('tarp_f (K=10)',    'tarp_f'),
    ('mira_scores',      'mira_scores'),
    ('top1_rmsd',        'top1_rmsd'),
    ('pb_json',          'pb_json'),
    ('tarp_translation', 'tarp_translation'),
    ('tarp_rotation',    'tarp_rotation'),
    ('tarp_torsion',     'tarp_torsion'),
    ('mira_translation', 'mira_translation'),
    ('mira_rotation',    'mira_rotation'),
    ('mira_torsion',     'mira_torsion'),
]
cond_labels = [CONDITIONS[c]['label'].replace('SigmaDock ', '') for c in CONDITIONS]
print(f"{'':22s}  {'DD':>4}  " + '  '.join(f'{lb:>10}' for lb in cond_labels))
print('─' * (22 + 4 + 4 + len(CONDITIONS) * 12))
for label, key in keys:
    row = f"{label:22s}  {_r(dd.get(key)):>4}  "
    row += '  '.join(f"{_r(sd[c].get(key)):>10}" for c in CONDITIONS)
    print(row)

## 3. Top-1 RMSD accuracy

In [ ]:
def _accuracy(top1_rmsd, pb_json):
    """Return (rmsd_frac, pb_rmsd_frac) or (val, None) if pb_json missing."""
    if top1_rmsd is None:
        return None, None
    rmsd_frac = float((top1_rmsd < 2.0).mean())
    if pb_json is None:
        return rmsd_frac, None
    n = len(pb_json)
    # valid if rank-1 is in valid_ranks
    pb_and_rmsd = sum(
        1 for k, v in pb_json.items()
        if v.get('valid_ranks') and v['valid_ranks'][0] == 'rank1'
    ) / n
    return rmsd_frac, pb_and_rmsd

# Collect
acc = {}
for cond in CONDITIONS:
    acc[cond] = _accuracy(sd[cond]['top1_rmsd'], sd[cond]['pb_json'])
dd_acc = _accuracy(dd['top1_rmsd'], None)   # DiffDock PB json uses different format

# Plot
models  = list(CONDITIONS.keys()) + ['diffdock']
labels  = [CONDITIONS[c]['label'] for c in CONDITIONS] + [DD_LABEL]
colors  = [CONDITIONS[c]['color'] for c in CONDITIONS] + [DD_COLOR]
rmsd_vals = [acc[c][0] for c in CONDITIONS] + [dd_acc[0]]
pb_vals   = [acc[c][1] for c in CONDITIONS] + [None]

x = np.arange(len(models))
w = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
bars1 = ax.bar(x - w/2, [v if v is not None else 0 for v in rmsd_vals],
               w, color=colors, alpha=0.9, label='RMSD < 2 Å')
bars2 = ax.bar(x + w/2, [v if v is not None else 0 for v in pb_vals],
               w, color=colors, alpha=0.4, label='RMSD < 2 Å + PB valid',
               hatch='//')

for bar, val in zip(bars1, rmsd_vals):
    if val is not None:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, pb_vals):
    if val is not None:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Fraction of complexes')
ax.set_ylim(0, 1.0)
ax.set_title('Top-1 RMSD accuracy (heuristic ranking)')
ax.legend(loc='lower right')
ax.axhline(1.0, color='k', lw=0.5, ls=':')
fig.tight_layout()
plt.show()

## 4. Flat TARP — symRMSD K=10 ECP curves

Each panel: one SigmaDock condition (solid) vs DiffDock (dashed red) as reference.
DiffDock uses K=10 if available, otherwise falls back to K=1.

In [ ]:
def _ecp_band(ax, fracs, label, color, ls='-', n_boot=N_BOOT):
    """Compute ECP + bootstrap CI and plot on ax. Returns True if plotted."""
    if fracs is None:
        return False
    ecp, alpha = ecp_from_fractions(fracs)
    boot = bootstrap_ecp(fracs, n_bootstrap=n_boot)
    plot_ecp(ecp, alpha, ax=ax, label=label, color=color,
             bootstrap_ecps=boot, linestyle=ls)
    return True

# Determine DiffDock TARP to use
dd_tarp = dd['tarp_f'] if dd['tarp_f'] is not None else dd['tarp_K1']
dd_tarp_label = DD_LABEL + (' (K=10)' if dd['tarp_f'] is not None else ' (K=1)')

# Check which SD conditions have data
conds_ready = [c for c in CONDITIONS if sd[c]['tarp_f'] is not None]
if not conds_ready:
    print('No SigmaDock TARP data yet — run mira_tarp jobs first.')
else:
    ncols = len(conds_ready)
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5), sharey=True)
    if ncols == 1:
        axes = [axes]

    for ax, cond in zip(axes, conds_ready):
        cfg = CONDITIONS[cond]
        _ecp_band(ax, sd[cond]['tarp_f'], cfg['label'], cfg['color'])
        _ecp_band(ax, dd_tarp, dd_tarp_label, DD_COLOR, ls=DD_LS)
        ax.set_title(cfg['label'])
        ax.set_xlabel('Coverage threshold α')
        if ax is axes[0]:
            ax.set_ylabel('Empirical coverage')

    fig.suptitle('TARP ECP — symRMSD K=10  (diagonal = perfect calibration)', y=1.01)
    fig.tight_layout()
    plt.show()

In [ ]:
# Overlay: all conditions + DiffDock in one panel
if conds_ready:
    fig, ax = plt.subplots(figsize=(6, 5))
    for cond in conds_ready:
        cfg = CONDITIONS[cond]
        _ecp_band(ax, sd[cond]['tarp_f'], cfg['label'], cfg['color'])
    _ecp_band(ax, dd_tarp, dd_tarp_label, DD_COLOR, ls=DD_LS)
    ax.set_xlabel('Coverage threshold α')
    ax.set_ylabel('Empirical coverage')
    ax.set_title('TARP ECP — all conditions overlaid')
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()

## 5. Flat MIRA

Left: violin per model. Right: mean MIRA per condition with ±1σ error bars.
Null reference line at `mira_null(40) ≈ 0.683`.

In [ ]:
mira_ready = {c: sd[c]['mira_scores'] for c in CONDITIONS if sd[c]['mira_scores'] is not None}
if not mira_ready and dd['mira_scores'] is None:
    print('No MIRA data yet.')
else:
    # Build dataset list: SigmaDock conditions + DiffDock
    datasets = [(CONDITIONS[c]['label'], sd[c]['mira_scores'], CONDITIONS[c]['color'])
                for c in CONDITIONS if sd[c]['mira_scores'] is not None]
    if dd['mira_scores'] is not None:
        datasets.append((DD_LABEL, dd['mira_scores'], DD_COLOR))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: violin ─────────────────────────────────────────────────────────
    ax = axes[0]
    positions = np.arange(len(datasets))
    for pos, (lbl, scores, col) in zip(positions, datasets):
        vp = ax.violinplot([scores], positions=[pos], showmedians=True,
                           showextrema=False, widths=0.7)
        vp['bodies'][0].set_facecolor(col)
        vp['bodies'][0].set_alpha(0.65)
        vp['cmedians'].set_color(col)
        vp['cmedians'].set_linewidth(2)
    ax.axhline(MIRA_NULL, color='k', ls='--', lw=1.2, label=f'null ({MIRA_NULL:.3f})')
    ax.set_xticks(positions)
    ax.set_xticklabels([d[0] for d in datasets], rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('MIRA score')
    ax.set_title('MIRA score distribution')
    ax.legend(fontsize=9)

    # ── Right: mean ± std bar chart ──────────────────────────────────────────
    ax = axes[1]
    means  = [s.mean() for _, s, _ in datasets]
    stds   = [s.std()  for _, s, _ in datasets]
    colors = [c        for _, _, c in datasets]
    bars = ax.bar(positions, means, color=colors, alpha=0.85,
                  yerr=stds, capsize=4, error_kw={'ecolor': 'k', 'lw': 1.2})
    ax.axhline(MIRA_NULL, color='k', ls='--', lw=1.2, label=f'null ({MIRA_NULL:.3f})')
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.002,
                f'{m:.4f}', ha='center', va='bottom', fontsize=8)
    delta = [m - MIRA_NULL for m in means]
    for pos, m, d in zip(positions, means, delta):
        ax.text(pos, m/2, f'Δ={d:+.4f}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
    ax.set_xticks(positions)
    ax.set_xticklabels([d[0] for d in datasets], rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Mean MIRA score')
    ax.set_title('Mean MIRA ± 1σ  (Δ = deviation from null)')
    ax.legend(fontsize=9)

    fig.tight_layout()
    plt.show()

## 6. Group TARP — symRMSD K=100

One figure per DOF (Translation / Rotation / Torsion).  
Each figure: one panel per SigmaDock condition + one overlay panel.

In [ ]:
for g in GROUPS:
    conds_g = [c for c in CONDITIONS if sd[c][f'tarp_{g}'] is not None]
    dd_g    = dd[f'tarp_{g}']

    if not conds_g and dd_g is None:
        print(f'Group TARP {g}: no data yet.')
        continue

    # Individual panels + one overlay
    ncols = len(conds_g) + 1   # +1 for overlay
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5), sharey=True)
    if ncols == 1:
        axes = [axes]

    for ax, cond in zip(axes[:-1], conds_g):
        cfg = CONDITIONS[cond]
        _ecp_band(ax, sd[cond][f'tarp_{g}'], cfg['label'], cfg['color'])
        _ecp_band(ax, dd_g, DD_LABEL, DD_COLOR, ls=DD_LS)
        ax.set_title(cfg['label'], fontsize=10)
        ax.set_xlabel('α')
        if ax is axes[0]:
            ax.set_ylabel('Empirical coverage')

    # Overlay panel
    ax_ov = axes[-1]
    for cond in conds_g:
        cfg = CONDITIONS[cond]
        _ecp_band(ax_ov, sd[cond][f'tarp_{g}'], cfg['label'], cfg['color'])
    _ecp_band(ax_ov, dd_g, DD_LABEL, DD_COLOR, ls=DD_LS)
    ax_ov.set_title('All conditions', fontsize=10)
    ax_ov.set_xlabel('α')
    ax_ov.legend(fontsize=7, loc='upper left')

    fig.suptitle(f'Group TARP — {GROUP_LABELS[g]}  (K=100)', y=1.01, fontsize=12)
    fig.tight_layout()
    plt.show()

## 7. Group MIRA

Grouped bar chart: x-axis = DOF, bars per model condition + DiffDock.  
Null reference line per panel.

In [ ]:
# Collect group MIRA means
all_models = list(CONDITIONS.keys()) + ['diffdock']
all_labels = [CONDITIONS[c]['label'] for c in CONDITIONS] + [DD_LABEL]
all_colors = [CONDITIONS[c]['color'] for c in CONDITIONS] + [DD_COLOR]

group_means = {m: {} for m in all_models}
group_stds  = {m: {} for m in all_models}
group_ns    = {m: {} for m in all_models}

for cond in CONDITIONS:
    for g in GROUPS:
        arr = sd[cond][f'mira_{g}']
        group_means[cond][g] = arr.mean() if arr is not None else None
        group_stds[cond][g]  = arr.std()  if arr is not None else None
        group_ns[cond][g]    = len(arr)   if arr is not None else 0

for g in GROUPS:
    arr = dd[f'mira_{g}']
    group_means['diffdock'][g] = arr.mean() if arr is not None else None
    group_stds['diffdock'][g]  = arr.std()  if arr is not None else None
    group_ns['diffdock'][g]    = len(arr)   if arr is not None else 0

any_data = any(group_means[m][g] is not None
               for m in all_models for g in GROUPS)
if not any_data:
    print('No group MIRA data yet.')
else:
    n_models = len(all_models)
    x = np.arange(len(GROUPS))
    w = 0.75 / n_models
    offsets = np.linspace(-(n_models-1)*w/2, (n_models-1)*w/2, n_models)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (model, lbl, col) in enumerate(zip(all_models, all_labels, all_colors)):
        means = [group_means[model][g] for g in GROUPS]
        stds  = [group_stds[model][g]  for g in GROUPS]
        valid = [m is not None for m in means]
        xpos  = x[valid] + offsets[i]
        m_arr = [m for m, v in zip(means, valid) if v]
        s_arr = [s for s, v in zip(stds,  valid) if v]
        if m_arr:
            ax.bar(xpos, m_arr, w * 0.9, color=col, alpha=0.85,
                   label=lbl, yerr=s_arr, capsize=3,
                   error_kw={'ecolor': 'k', 'lw': 1})

    ax.axhline(MIRA_NULL, color='k', ls='--', lw=1.2, label=f'null ({MIRA_NULL:.3f})')
    ax.set_xticks(x)
    ax.set_xticklabels([GROUP_LABELS[g] for g in GROUPS])
    ax.set_ylabel('Mean MIRA score')
    ax.set_title('Group MIRA — mean ± 1σ per DOF  (K=100)')
    ax.legend(fontsize=9, loc='lower right')
    ax.set_ylim(bottom=min(MIRA_NULL - 0.05,
                           min((v for m in all_models for v in group_means[m].values()
                                if v is not None), default=MIRA_NULL) - 0.02))
    fig.tight_layout()
    plt.show()

## 8. Summary table

In [ ]:
from molcalib.tarp import atc_score

def _tarp_auc(fracs):
    if fracs is None:
        return None
    ecp, alpha = ecp_from_fractions(fracs)
    return float(np.trapz(ecp, alpha))

def _fmt(v, fmt='.4f'):
    return f'{v:{fmt}}' if v is not None else '   –'

HEADER = (f"{'Model':<22}  {'N':>5}  {'RMSD<2Å':>8}  {'MIRA':>8}  "
          f"{'ΔMIRA':>7}  {'TARP AUC':>9}  "
          f"{'MIRA_tr':>8}  {'MIRA_rot':>9}  {'MIRA_tor':>9}")
print(HEADER)
print('─' * len(HEADER))

all_entries = [(c, CONDITIONS[c]['label'], sd[c]) for c in CONDITIONS]
all_entries.append(('diffdock', DD_LABEL, dd))

for key, label, data in all_entries:
    ms    = data['mira_scores']
    mira  = ms.mean() if ms is not None else None
    delta = (mira - MIRA_NULL) if mira is not None else None
    n     = len(ms) if ms is not None else (len(data['top1_rmsd']) if data['top1_rmsd'] is not None else None)
    rmsd  = float((data['top1_rmsd'] < 2.0).mean()) if data['top1_rmsd'] is not None else None
    auc   = _tarp_auc(data.get('tarp_f'))
    m_tr  = data[f'mira_translation'].mean() if data.get('mira_translation') is not None else None
    m_rot = data[f'mira_rotation'].mean()    if data.get('mira_rotation')    is not None else None
    m_tor = data[f'mira_torsion'].mean()     if data.get('mira_torsion')     is not None else None

    print(f"{label:<22}  {str(n) if n else '–':>5}  "
          f"{_fmt(rmsd, '.1%'):>8}  {_fmt(mira):>8}  "
          f"{_fmt(delta,'+.4f'):>7}  {_fmt(auc):>9}  "
          f"{_fmt(m_tr):>8}  {_fmt(m_rot):>9}  {_fmt(m_tor):>9}")

print()
print(f"MIRA null (S=40): {MIRA_NULL:.4f}  |  TARP AUC perfect: 0.5000")